# 随机退火算法
- 物理上，金属加热到高温后缓慢冷却，原子会逐渐排列成能量最低的稳定晶体结构。如果冷却太快，则会停留在能量较高的非晶态。
- 模拟退火把这一过程类比到优化问题中：
> 1. 解（solution）对应物理系统的状态；
> 2. 目标函数值（cost）对应能量；
> 3. 温度 T 控制搜索的随机程度，温度越高越容易接受差解，温度降低后逐渐收敛。

- 关于下列代码中关键参数的说明
> 1. loss(x)（能量函数）——对应算法中的 $f(x)$。它把一个候选解映射为一个"代价"，SA 的目标是让它尽量小。
> 2. start——初始状态
> 3. cur_state——当前状态
> 4. temperature——控制随机程度。它只影响接受概率，温度越高随机性越强——需要设置一个下限
> 5. cooling——每次随机探索后都要降温
> 6. delta——能量差——当前状态的loss - 下次备选状态的loss——这个能量差
> > 1）决定了是否采用备选状态
> >
> > 2）影响了随机的程度

In [1]:
import math
import torch


def simulated_annealing(
    coefficients: torch.Tensor,
    start: float,
    initial_temperature: float,
    cooling: float,
    offsets: torch.Tensor,
    uniforms: torch.Tensor,
) -> torch.Tensor:

    powers = torch.arange(len(coefficients))
    def loss(x):
        return (coefficients * x ** powers).sum()

    cur_state = torch.tensor(start, dtype=coefficients.dtype)
    best_state = cur_state
    best_loss = loss(cur_state)

    temperature = initial_temperature

    current_loss = loss(cur_state)
    for i in range(len(offsets)):
        # 1.计算备选值candidate
        candidate = cur_state + offsets[i]
        candidate_loss = loss(candidate)
        # 2.计算loss(candidate) - loss(cur_state)即
        delta = candidate_loss - current_loss

        # 3.判断能否取用这个candidate
        if delta <= 0 or uniforms[i] < math.exp(-delta / temperature):
            cur_state = candidate
            current_loss = candidate_loss
            if current_loss < best_loss:
                best_state, best_loss = cur_state.clone(), current_loss.clone()
        # 4.更新温度
        temperature = max(temperature * cooling, 1e-12)
    return torch.stack([best_state, best_loss])

In [4]:
coefficients = torch.tensor([37,0,-12,-4,3])
start = -1
initial_temperature = 10
cooling = 0.9
offsets = torch.tensor([1,1,1,-0.5])
uniforms = torch.tensor([0.4,0.7,0.2,0.1])

# 预期结果 [2.0,5.0]
simulated_annealing(coefficients, start, initial_temperature, cooling, offsets, uniforms)

tensor([2., 5.])